# 04 — Results, claim checks, and manuscript figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanturan/apsp-knn-benchmark/blob/main/notebooks/04_results_and_figures.ipynb)

This notebook uses the **archived benchmark outputs** from the reported session. It is the fastest way to verify the manuscript values and regenerate analysis figures.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_NAME = "apsp-knn-benchmark"
REPO_URL = "https://github.com/gokhanturan/apsp-knn-benchmark.git"

# Colab: clone the repository if the notebook was opened directly from GitHub.
if Path('/content').exists() and not (Path('/content') / REPO_NAME).exists():
    subprocess.run(['git', 'clone', REPO_URL, str(Path('/content') / REPO_NAME)], check=True)

if (Path('/content') / REPO_NAME).exists():
    ROOT = Path('/content') / REPO_NAME
else:
    # Local/Jupyter execution from repo/notebooks or repo root.
    cwd = Path.cwd().resolve()
    ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

os.chdir(ROOT)
print('Repository root:', ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import sys, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

R = ROOT / 'results'
FOUT = ROOT / 'reproduced_figures'
FOUT.mkdir(exist_ok=True)

runtime = pd.read_csv(R / 'runtime_summary_30.csv')
memory = pd.read_csv(R / 'memory_summary_30.csv')
scaling_seed = pd.read_csv(R / 'scaling_seed_summary.csv')
reversal = pd.read_csv(R / 'scaling_order_reversal.csv')


## Current-manuscript claim checks

In [ ]:
subprocess.run([sys.executable, 'src/check_current_manuscript_claims.py'], check=True)
claims = pd.read_csv(R / 'current_manuscript_claim_checks.csv')
display(claims)
assert claims['pass'].all()
print(f"Passed: {int(claims['pass'].sum())}/{len(claims)}")

## Table 4 core values and speedup

In [ ]:
p = runtime.pivot_table(index=['dataset','k'], columns='algorithm', values='wall_median_s')
p['dijkstra_speedup'] = p['floyd_warshall'] / p['dijkstra']
p['fastest'] = p[['floyd_warshall','johnson','dijkstra']].idxmin(axis=1)
display(p)
print('Median Dijkstra speedup:', round(float(p.dijkstra_speedup.median()), 4))
print('Maximum Dijkstra speedup:', round(float(p.dijkstra_speedup.max()), 4))

## Figure 1 — Run-level wall-clock distributions

In [ ]:
raw = pd.read_csv(R / 'runtime_raw_30.csv')
raw['condition'] = raw['dataset'].str.replace('_',' ', regex=False) + ' k=' + raw['k'].astype(str)
conditions = list(dict.fromkeys(raw['condition']))
algorithms = ['floyd_warshall','johnson','dijkstra']
fig, ax = plt.subplots(figsize=(15,6))
positions=[]; data=[]; labels=[]
pos=1
for c in conditions:
    for a in algorithms:
        vals=raw[(raw.condition==c)&(raw.algorithm==a)].wall_time_s.to_numpy()
        data.append(vals); positions.append(pos); labels.append(a.replace('_',' ')); pos+=1
    pos+=0.8
ax.boxplot(data, positions=positions, widths=0.65, showfliers=False)
ax.set_yscale('log')
ax.set_ylabel('Wall-clock time (s)')
centers=[]
pos=1
for c in conditions:
    centers.append(pos+1); pos+=3.8
ax.set_xticks(centers); ax.set_xticklabels(conditions, rotation=45, ha='right')
ax.set_title('Run-level wall-clock distributions')
fig.tight_layout()
fig.savefig(FOUT/'figure1_runtime_distributions.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 2 — Speedup relative to Floyd-Warshall

In [ ]:
summary = runtime.pivot_table(index=['dataset','k'], columns='algorithm', values='wall_median_s').reset_index()
summary['Dijkstra'] = summary['floyd_warshall']/summary['dijkstra']
summary['Johnson'] = summary['floyd_warshall']/summary['johnson']
fig, ax = plt.subplots(figsize=(10,5))
for dataset, g in summary.groupby('dataset'):
    g=g.sort_values('k')
    ax.plot([f"{dataset} k={k}" for k in g.k], g['Dijkstra'], marker='o', label=f'{dataset}: Dijkstra')
    ax.plot([f"{dataset} k={k}" for k in g.k], g['Johnson'], marker='x', linestyle='--', label=f'{dataset}: Johnson')
ax.axhline(1.0, linewidth=1)
ax.set_ylabel('Speedup vs Floyd-Warshall')
ax.tick_params(axis='x', rotation=60)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig(FOUT/'figure2_speedup.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 3 — Runtime vs process-level incremental peak RSS

In [ ]:
m = memory[['dataset','k','algorithm','incremental_peak_rss_median_mib']]
r = runtime[['dataset','k','algorithm','wall_median_s']]
rm = r.merge(m, on=['dataset','k','algorithm'])
fig, ax = plt.subplots(figsize=(8,6))
for alg, g in rm.groupby('algorithm'):
    ax.scatter(g['incremental_peak_rss_median_mib'], g['wall_median_s'], label=alg.replace('_',' '))
ax.set_yscale('log')
ax.set_xlabel('Incremental peak RSS (MiB)')
ax.set_ylabel('Median wall-clock time (s)')
ax.legend()
fig.tight_layout()
fig.savefig(FOUT/'figure3_runtime_memory.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 4 — Controlled within-Digits scaling

In [ ]:
s = scaling_seed.groupby(['n','k','algorithm'], as_index=False).agg(median=('wall_median_s','median'))
fig, ax = plt.subplots(figsize=(9,6))
for (k,alg), g in s[s.algorithm.isin(['floyd_warshall','dijkstra'])].groupby(['k','algorithm']):
    g=g.sort_values('n')
    ax.plot(g['n'], g['median'], marker='o', label=f'k={k} {alg.replace("_"," ")}')
ax.set_yscale('log')
ax.set_xlabel('Number of vertices')
ax.set_ylabel('Median wall-clock time (s)')
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig(FOUT/'figure4_controlled_scaling.png', dpi=200, bbox_inches='tight')
plt.show()
display(reversal)

The repository also contains the original manuscript figures in `figures/`. The regenerated plots above are intended as transparent analysis reproductions; minor visual styling differences do not affect the reported numerical results.